In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
import scipy
from scipy.io import loadmat
import mne

In [2]:
import sys
import os

sys.path.append(r'C:\Users\torho\PycharmProjects\TranformerP300')

In [3]:
data = scipy.io.loadmat('data_kirasirova/S1901-P300_classic.mat')
X_real = data['epochs']
y_real = data['labels'].flatten()

In [5]:
from generate_data import build_sequences_new

In [6]:
ch_names = ['Fz', 'Cz', 'Pz']
sfreq = 250
info = mne.create_info(ch_names, sfreq, ch_types='eeg')
events = np.column_stack((np.arange(len(y_real)), np.zeros(len(y_real), int), y_real))
event_id = {'non-target': 0, 'target': 1}
epochs_mne = mne.EpochsArray(X_real, info, events=events, event_id=event_id)
epochs_mne.filter(l_freq=1, h_freq=15, filter_length='auto')

all_target = epochs_mne['target']
all_non_target = epochs_mne['non-target']

synth_sequences, synth_targets = build_sequences_new(
    all_target, all_non_target, n_classes=16, n_samples=2000
)


Not setting metadata
6759 matching events found
No baseline correction applied
0 projection items activated
Setting up band-pass filter from 1 - 15 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 15.00 Hz
- Upper transition bandwidth: 3.75 Hz (-6 dB cutoff frequency: 16.88 Hz)
- Filter length: 825 samples (3.300 s)



C:\Users\torho\AppData\Local\Temp\ipykernel_49960\563296646.py:7: RuntimeWarning: filter_length (825) is longer than the signal (250), distortion is likely. Reduce filter length or filter a longer signal.
  epochs_mne.filter(l_freq=1, h_freq=15, filter_length='auto')


In [7]:
synth_sequences.shape

(2000, 16, 3, 250)

In [10]:

batch_size = synth_sequences.shape[0]
X_collapsed = synth_sequences.reshape(batch_size, 1, 3, 16 * 250)  #в (2000, 1, 3, 4000)
X_collapsed.shape



(2000, 1, 3, 4000)

In [11]:
y_tensor = torch.LongTensor(synth_targets)

In [12]:
class SimpleTransformerCollapsed(nn.Module):
    def __init__(self, n_classes=16):
        super().__init__()
        
        #Пространственная свертка
        # (Batch, 1, 3, 4000) ->(Batch, 32, 1, 4000)
        self.spatial_conv = nn.Conv2d(1, 32, (3, 1), bias=False)
        self.bn1 = nn.BatchNorm2d(32)

        #Временная свертка (Batch, 32, 1, 4000) -> (Batch, 32, 1, 1000)
        self.temporal_conv = nn.Conv2d(32, 32, (1, 8), stride=(1, 4), padding=(0, 2))
        self.bn2 = nn.BatchNorm2d(32)

        #Пуллинг (Batch, 32, 1, 1000)->(Batch, 32, 1, 250)
        self.pool = nn.AdaptiveAvgPool2d((1, 250))
        
        self.pre_norm = nn.LayerNorm(32)
        self.pre_dropout = nn.Dropout(0.1)
        self.pre_linear = nn.Linear(32, 32)
        
        #pos. encoding
        self.pos_encoder = nn.Parameter(torch.randn(1, 250, 32) * 0.02)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=32, nhead=4, dim_feedforward=128,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.classifier = nn.Linear(32, n_classes)
        
    def forward(self, x):
        x = self.spatial_conv(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        
        x = self.temporal_conv(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)
        
        x = self.pool(x)                    
        x = x.squeeze(2)
        x = x.permute(0, 2, 1)
        
        x = self.pre_norm(x)
        x = self.pre_dropout(x)
        x = self.pre_linear(x)
        x = F.gelu(x)
        x = x + self.pos_encoder[:, :x.size(1), :]
        
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.classifier(x)

In [15]:
#санити чек
X_small = X_collapsed[:10]
y_small = y_tensor[:10]

X_small_tensor = torch.FloatTensor(X_small)
y_small_tensor = torch.LongTensor(y_small)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleTransformerCollapsed(n_classes=16).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

X_small_tensor = X_small_tensor.to(device)
y_small_tensor = y_small_tensor.to(device)

for epoch in range(100):
    optimizer.zero_grad()
    output = model(X_small_tensor)
    loss = criterion(output, y_small_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}, Loss: {loss.item():.6f}")

with torch.no_grad():
    output = model(X_small_tensor)
    pred = output.argmax(dim=1)
    acc = (pred == y_small_tensor).float().mean().item()
    print(f"точность {acc*100:.1f}%")
    print(f"предсказания {pred.cpu().tolist()}")
    print(f"истинные метки    {y_small_tensor.cpu().tolist()}")

Epoch  20, Loss: 2.107248
Epoch  40, Loss: 1.433564
Epoch  60, Loss: 0.743382
Epoch  80, Loss: 0.335416
Epoch 100, Loss: 0.173529
точность 100.0%
предсказания [0, 11, 8, 6, 14, 13, 11, 12, 2, 3]
истинные метки    [0, 11, 8, 6, 14, 13, 11, 12, 2, 3]
